# PyHoloscope Tutorial 2: Off-Axis Holography

In off-axis holographic microscopy, light that has passed through a sample interferes on the camera with a slightly tilted reference beam. This tilt introduces a spatial carrier modulation in the recorded hologram. In the spatial frequency (Fourier) domain, this modulation creates two sidebands that are shifted away from the origin by the carrier frequency. By isolating one of these shifted components and transforming it back to the spatial domain, the complex optical field is obtained, allowing the phase of the sample to be recovered. This notebook demonstrates off-axis image recovery using PyHoloscope.

(If you are not familiar with Juypter notebooks you can select *Run -> Run All Cells* to see the output).

In [ ]:
from matplotlib import pyplot as plt
import sys; sys.path.append('..\src')   # Allows us to find PyHoloscope if not pip installed
import pyholoscope as pyh

An example hologram is saved as a tif file - this is what is captured by the camera. We can load this saved hologram using a convenience function in PyHoloscope, and then we display it (the paramecium is barely visible in the raw hologram):

In [ ]:
hologram = pyh.load_image(r"../example_data/off_axis_paramecium/in_focus.tif")

plt.figure(dpi=100, figsize = (3,3)); plt.imshow(hologram, cmap='gray')

The modulation can be seen if we zoom into a small area:

In [ ]:
plt.figure(dpi=100, figsize = (3,3)); plt.imshow(hologram[100:150,100:150], cmap='gray')

## Demodulation and Phase Recovery
To demodulate and recover the phase, we need to know what the spatial frequency of the modulation is. PyHoloscope can calculate this either from the image or, ideally, a background image with nothing in the field of view.

In [ ]:
background = pyh.load_image(r"../example_data/off_axis_paramecium/background.tif")

holo = pyh.Holo(
    mode=pyh.OFF_AXIS,
    background=background,  
) 

holo.calib_off_axis()  


We then reconstruct the hologram and display the phase:

In [ ]:
recon_field = holo.process(hologram)

plt.figure(dpi=100, figsize = (3,3))
plt.title("Phase")
plt.imshow(pyh.phase(recon_field), cmap="twilight", interpolation="none")



## Phase Correction
This reconstruction shows a background phase due to imperfectins in the imaging system. This can be corrected using the background image by setting ``relative_phase``:


In [ ]:
holo = pyh.Holo(
    mode=pyh.OFF_AXIS,
    background=background,  # For correcting background phase
    relative_phase=True,
)  # We will remove the background phase

holo.calib_off_axis()  # Finds modulation frequency and
# pre-computes background phase

# Remove the off-axis modulation and recover the phase
recon_field = holo.process(hologram)


plt.figure(dpi=100, figsize = (3,3)); plt.title("Phase")
plt.imshow(pyh.phase(recon_field), cmap="twilight", interpolation="none")

## Tilt Removal
There is still a tilt of the phase due to the slide or coverslip; as this was not present in the background image it was not removed. This tilt can be detected and removed:

In [ ]:

phase_unwrapped = pyh.phase_unwrap(pyh.phase(recon_field))

tilt = pyh.obtain_tilt(phase_unwrapped)

phase_untilted = pyh.relative_phase(phase_unwrapped, tilt)

plt.figure(dpi=100, figsize = (3,3)); plt.title("Phase")
plt.imshow(phase_untilted, cmap="twilight", interpolation="none")

## Phase Visualisation
We can also generate synthetic DIC images:


In [ ]:
DIC = pyh.synthetic_DIC(recon_field)

plt.figure(dpi=100, figsize = (3,3)); plt.title("Synthetic DIC")
plt.imshow(DIC, cmap="gray", interpolation="none")

Or phase gradient:

In [ ]:
phase_grad = pyh.phase_gradient(phase_untilted)

plt.figure(dpi=100, figsize = (3,3)); plt.title("Phase Gradient")
plt.imshow(phase_grad, cmap="gray", interpolation="none")